# Healthcare Agent Learning Lab 2/10

## The minimal agent loop

**Suggested time:** 20 minutes  
**Demonstration type:** Scripted or live manual agent loop

### Learning objectives

- Recognize the perceive–decide–act–observe loop.
- Separate an agent trajectory from a single model response.
- Identify stop conditions, tool boundaries, and failure points.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** make the *control loop* visible. A single agent with a few tools works a recurring clinical task: **post-discharge anticoagulant follow-up.** It reads state, reasons, drafts an action, and stops — it does **not** change any order itself.

No framework is used on purpose — the loop is ~30 lines so you can see exactly what an 'agent' is.


> ### In plain terms
> An "agent" sounds fancy but is just a model run in a **loop**: look at the situation → decide one action → do it → look again — until it's finished. Here it handles a real clinical task (a blood-thinner follow-up) in about 30 lines of ordinary code, so you can see there's no magic.
>
> **Notice:** it only ever *drafts a note for a human*. It never changes a medication itself — that boundary is the subject of the next notebook.
>
> **What you'll see:** numbered turns of the loop. `PLAN` = what it decided this turn · `ACT` = the tool it used · `OBS` = what came back.


In [ ]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    """Scripted ReAct actions for the anticoagulant case, driven by which tools
    have already run (visible in the message history)."""
    hist = " ".join(m["content"] for m in messages if m["role"]!="system")
    def done(tool): return f'"observation for {tool}"' in hist or f"[{tool}]" in hist
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Start by reading the latest INR.","action":"get_latest_inr","action_input":{}})
    if "get_medications" not in hist:
        return json.dumps({"thought":"INR looks high; check the medication list.","action":"get_medications","action_input":{}})
    if "draft_followup" not in hist:
        return json.dumps({"thought":"INR 4.2 is above the 2-3 range on warfarin. Draft a dose-review flag and earlier follow-up.",
                           "action":"draft_followup","action_input":{"text":"INR 4.2 (above 2-3) on warfarin. Flag for dose review; bring follow-up forward to 3 days."}})
    return json.dumps({"thought":"Handoff drafted for the clinician.","action":"final",
                       "action_input":{"summary":"Drafted a dose-review flag + earlier follow-up. No order changed."}})


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### The environment: tools the agent can use
A tiny synthetic patient. Read-tools return state; the draft-tool proposes text for a human. (Swap these for real FHIR calls later — the loop is unchanged.)


In [ ]:
PATIENT = {"inr": 4.2, "range": "2.0-3.0",
           "medications": ["warfarin 5 mg daily", "lisinopril 10 mg daily"],
           "notes": "Discharged 5 days ago after DVT; started on warfarin."}
DRAFTS = []

def _text_arg(a, *names):
    "Pull the intended text no matter which key the model chose for it."
    for n in names:
        if a.get(n): return a[n]
    for v in a.values():
        if isinstance(v, str) and v.strip(): return v
    return ""

# Every tool takes **any keyword. A live model routinely invents or renames
# arguments; a tool that hard-codes its parameter names would crash on them.
def get_latest_inr(**_):    return {"inr": PATIENT["inr"], "reference_range": PATIENT["range"]}
def get_medications(**_):   return {"medications": PATIENT["medications"]}
def get_recent_notes(**_):  return {"notes": PATIENT["notes"]}
def draft_followup(**a):
    text = _text_arg(a, "text","followup_draft","summary","note","content","message")
    # Some models split the intended draft into several named fields. Preserve
    # those fields in a readable order; never guess from an arbitrary string,
    # which could be a lab value or reference range.
    if not text:
        parts = [a.get(name) for name in ("dose_review_flag", "follow_up", "reason", "flag")]
        text = " ".join(value for value in parts if isinstance(value, str) and value.strip())
    if not text:
        text = json.dumps(a, ensure_ascii=False, sort_keys=True)
    DRAFTS.append(text); return {"drafted": text}

TOOLS = {"get_latest_inr":get_latest_inr, "get_medications":get_medications,
         "get_recent_notes":get_recent_notes, "draft_followup":draft_followup}


### The loop
The model returns JSON `{thought, action, action_input}` each turn. We **act** (run the tool), feed the **observation** back, and repeat until it emits `action: "final"`.


In [ ]:
SYSTEM = ("You are a clinical follow-up assistant. Each turn, output ONLY JSON: "
          '{"thought":..., "action": one of ' + str(list(TOOLS)+["final"]) + ', "action_input": {{}}}. '
          "Read the INR and medications, and if the INR is out of range, DRAFT a dose-review flag and earlier "
          "follow-up for the clinician. Never change an order yourself. Then action=final.")

def parse(txt):
    """Parse the first JSON object and tolerate surrounding model commentary."""
    t = re.sub(r"^```[a-z]*|```$", "", txt.strip(), flags=re.M).strip()
    try:
        value = json.loads(t)
    except json.JSONDecodeError:
        start = t.find("{")
        if start < 0:
            raise ValueError(f"Model did not return a JSON object: {t[:200]!r}")
        value, _end = json.JSONDecoder().raw_decode(t[start:])
    if not isinstance(value, dict):
        raise ValueError("Model action must be a JSON object.")
    allowed_fields = {"thought", "rationale", "action", "action_input"}
    unexpected = set(value) - allowed_fields
    if unexpected:
        raise ValueError(f"Unexpected action fields: {sorted(unexpected)}")
    action = value.get("action")
    if not isinstance(action, str) or not action:
        raise ValueError("Model action must contain a non-empty string 'action'.")
    action_input = value.get("action_input", {})
    if not isinstance(action_input, dict):
        raise ValueError("Model action_input must be a JSON object.")
    value["action_input"] = action_input
    available_tools = set(globals().get("TOOLS", {}))
    if available_tools and action not in available_tools | {"final"}:
        allowed = sorted(available_tools | {"final"})
        raise ValueError(f"Action {action!r} is not allowed; expected one of {allowed}.")
    if action in {"change_order", "write_flag"}:
        has_text = any(isinstance(item, str) and item.strip() for item in action_input.values())
        if not has_text:
            raise ValueError(f"Action {action!r} requires a non-empty textual argument.")
    if action == "draft_followup" and not action_input:
        raise ValueError("Action 'draft_followup' requires a non-empty structured payload.")
    return value

def run(system=SYSTEM, goal="Handle the anticoagulant follow-up.", max_steps=6):
    msgs=[{"role":"system","content":system},{"role":"user","content":goal}]
    for step in range(max_steps):
        act = parse(chat(msgs))
        action = act.get("action","final")
        print(f"\n[{step}] PLAN : {act.get('thought','')}")
        if action=="final":
            summary = act.get("action_input", {}).get("summary") or act.get("thought", "")
            print(f"     DONE : {summary}"); return
        tool = TOOLS.get(action)
        if tool is None:
            print(f"     (unknown action {action!r}; stopping)"); return
        obs  = tool(**act.get("action_input",{}))
        print(f"     ACT  : {action}({act.get('action_input',{})})")
        print(f"     OBS  : {obs}")
        msgs.append({"role":"assistant","content":json.dumps(act)})
        msgs.append({"role":"user","content":f"[{action}] observation: {json.dumps(obs)}"})

run()
print("\nDrafts awaiting clinician review:", DRAFTS)


### Takeaway
That's the whole idea of an agent: a model **iterating** over perceive→plan→act→observe against an environment. Notice the agent stopped at a *draft* — the consequential action (changing the dose) was never taken. NB 3 makes that boundary explicit with a human gate.


## Exercise

No programming is required for this section.

**Activity:** Read the printed trajectory and mark which step gathered evidence, which step interpreted it, and which step created a draft. Decide whether the available evidence was sufficient for the draft and where a clinician should enter the workflow.

### Discussion prompts

- What makes this an agent rather than a chatbot response?
- Could the agent stop too early or continue for too long?
- Which final state matters more than the wording of its answer?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: choose the safest next action after an incomplete record.
AVAILABLE_EVIDENCE = ["INR result", "medication list"]
MISSING_EVIDENCE = ["bleeding symptoms", "prescriber plan"]
NEXT_ACTION = "request_more_information"  # try: "draft_followup" or "change_order"
print("Evidence:", AVAILABLE_EVIDENCE)
print("Missing:", MISSING_EVIDENCE)
print("Selected next action:", NEXT_ACTION)
print("Boundary check:", "SAFE" if NEXT_ACTION != "change_order" else "CONSEQUENTIAL—requires an enforced gate")


## Optional technical extension

Add loop detection, a structured error state, and a maximum number of repeated calls to the same tool. Test an unknown tool and malformed model output.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

A loop requires state, observations, tool execution, and a stopping rule. A model emitting several paragraphs is not automatically an agent. The example drafts a handoff but does not change an order.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
